# Language Proficiency & Social Integration Among Non-English Speakers

---

## What is this notebook about?

This notebook collects and analyses data about **how well non-English speakers in the U.S. can speak English**, and how that relates to their social integration.

We use this as an **alternative/supplementary dataset** for our main research on: *"The Impact of Language Barriers on Social Integration and Self-Confidence among Migrants."*

---

## Where does the data come from?

- **Source:** U.S. Census Bureau — American Community Survey (ACS), 5-Year Estimates
- **Year:** 2022
- **API Link:** `https://api.census.gov/data/2022/acs/acs5`
- **Coverage:** ~3,000 U.S. counties
- **License:** Public Domain (free to use)

---

## What tables did we use from the Census?

| Table Code | What it contains |
|---|---|
| B16004 | Age × Language Spoken at Home × English Proficiency (people aged 18–64) |
| B14001 | School Enrollment by Level |
| B19013 | Median Household Income |
| B05012 | Native vs. Foreign-Born Population |

---

## What columns are in our final dataset?

| Column Name | What it means |
|---|---|
| county | Name of the U.S. county |
| state | Name of the U.S. state |
| state_encoded | State turned into a number (for machine learning) |
| total_non_english_speakers_18_64 | Adults (18–64) who speak a non-English language at home |
| foreign_born_population | People born outside the U.S. living in that county |
| spanish_speakers_total | Total Spanish speakers (18–64) |
| spanish_eng_very_well | Spanish speakers who speak English very well |
| spanish_eng_well | Spanish speakers who speak English well |
| spanish_eng_not_well | Spanish speakers who speak English not well |
| spanish_eng_not_at_all | Spanish speakers who speak no English |
| indo_euro_speakers_total | Indo-European language speakers (French, Hindi, Russian, etc.) |
| indo_euro_eng_very_well / well / not_well / not_at_all | Their English proficiency levels |
| asian_pac_speakers_total | Asian/Pacific Island language speakers (Mandarin, Korean, etc.) |
| asian_pac_eng_very_well / well / not_well / not_at_all | Their English proficiency levels |
| other_lang_speakers_total | Other language speakers (Arabic, Somali, Swahili, etc.) |
| other_lang_eng_very_well / well / not_well / not_at_all | Their English proficiency levels |
| total_enrolled_in_school | People enrolled in any school |
| enrolled_college_or_grad_school | People enrolled in college or graduate school |
| median_household_income_usd | Middle household income in US dollars |

---
## Step 1: Import the libraries we need

Think of libraries as toolboxes. Each one gives us specific tools:
- `requests` — lets us download data from the internet
- `BeautifulSoup` — helps us read and extract info from web pages (HTML)
- `pandas` — the main tool for working with tables of data
- `plotly` — creates interactive charts and graphs
- `LabelEncoder` — converts text labels (like state names) into numbers

In [1]:
# ── STEP 1: Import libraries ──

import requests                        # To download data from the Census API
from bs4 import BeautifulSoup          # To read HTML web pages
import pandas as pd                    # To work with data in table format
import warnings                        # To hide unnecessary warning messages
import plotly.express as px            # To create interactive charts easily
from sklearn.preprocessing import LabelEncoder  # To convert text to numbers

# Hide warning messages so the output stays clean
warnings.filterwarnings('ignore')

# Show more columns and wider text when printing tables
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)

print("All libraries loaded successfully!")

All libraries loaded successfully!


---
## Step 2: Set up the Census API details

The U.S. Census Bureau has a free API (a way for programs to request data).
We need to tell it:
- **Which year** of data we want (2022)
- **Which dataset** (ACS 5-Year Estimates)
- **Which columns** (called "variables") — each has a code like `B16004_025E`

Each code represents a specific piece of information. For example:
- `B16004_025E` = Total non-English speakers aged 18–64
- `B16004_026E` = Total Spanish speakers
- `B05012_003E` = Foreign-born population

In [2]:
# ── STEP 2: Set up API details ──

# The base URL for the Census API
BASE_URL = "https://api.census.gov/data"

# We want data from the year 2022
YEAR = 2022

# The specific dataset: American Community Survey, 5-Year Estimates
DATASET = "acs/acs5"

# List of all the variable codes we want to download
# Each code represents one column of data from the Census
variable_list = [
    "NAME",           # County and State name
    
    # --- Language Proficiency (Table B16004) ---
    "B16004_025E",    # Total non-English speakers (age 18-64)
    
    # Spanish speakers and their English levels
    "B16004_026E",    # Spanish speakers - total
    "B16004_027E",    # Spanish speakers - English very well
    "B16004_028E",    # Spanish speakers - English well
    "B16004_029E",    # Spanish speakers - English not well
    "B16004_030E",    # Spanish speakers - English not at all
    
    # Indo-European speakers (French, Hindi, Russian, etc.)
    "B16004_031E",    # Indo-European - total
    "B16004_032E",    # Indo-European - English very well
    "B16004_033E",    # Indo-European - English well
    "B16004_034E",    # Indo-European - English not well
    "B16004_035E",    # Indo-European - English not at all
    
    # Asian/Pacific Island speakers (Mandarin, Korean, Vietnamese, etc.)
    "B16004_036E",    # Asian/Pacific - total
    "B16004_037E",    # Asian/Pacific - English very well
    "B16004_038E",    # Asian/Pacific - English well
    "B16004_039E",    # Asian/Pacific - English not well
    "B16004_040E",    # Asian/Pacific - English not at all
    
    # Other language speakers (Arabic, Somali, Swahili, etc.)
    "B16004_041E",    # Other languages - total
    "B16004_042E",    # Other languages - English very well
    "B16004_043E",    # Other languages - English well
    "B16004_044E",    # Other languages - English not well
    "B16004_045E",    # Other languages - English not at all
    
    # --- School Enrollment (Table B14001) ---
    "B14001_002E",    # Total enrolled in school
    "B14001_008E",    # Enrolled in college or graduate school
    
    # --- Socioeconomic Indicators ---
    "B19013_001E",    # Median household income (USD)
    "B05012_003E",    # Foreign-born population
]

# Join all variable codes into one comma-separated string (the API needs it this way)
VARIABLES = ",".join(variable_list)

print(f"We are requesting {len(variable_list)} columns of data from the Census API.")

We are requesting 26 columns of data from the Census API.


---
## Step 3: Download the data from the Census API

This step does two things:
1. **Downloads the actual data** — one row for every U.S. county (~3,000 rows)
2. **Downloads human-readable labels** — so we know what each code means

The API returns data as JSON (a structured text format), which we convert into a pandas table (called a DataFrame).

In [3]:
# ── STEP 3: Download data from the Census API ──

# --- Part A: Download the main data ---

# Build the full URL for the API request
api_url = f"{BASE_URL}/{YEAR}/{DATASET}"

# Tell the API what we want:
#   - "get": which columns to download
#   - "for": get data for all counties
#   - "in": across all states
params = {
    "get": VARIABLES,
    "for": "county:*",
    "in": "state:*"
}

# Send the request to the Census API (wait up to 60 seconds)
response = requests.get(api_url, params=params, timeout=60)

# Check if the request was successful (will show an error if not)
response.raise_for_status()

# The API returns JSON data — convert it to a Python list
raw_data = response.json()

# The first row contains column headers, the rest is the actual data
# raw_data[0] = headers, raw_data[1:] = data rows
df_raw = pd.DataFrame(raw_data[1:], columns=raw_data[0])

print(f"Downloaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")


# --- Part B: Download human-readable labels for each variable code ---

# The Census has an HTML page listing what each variable code means
label_url = f"{BASE_URL}/{YEAR}/{DATASET}/variables.html"

# We'll store labels in a dictionary: {"B16004_025E": "description text", ...}
var_labels = {}

try:
    # Download the HTML page
    label_page = requests.get(label_url, timeout=20)
    
    # Parse the HTML so we can search through it
    soup = BeautifulSoup(label_page.text, "html.parser")
    
    # Find the table on the page
    table = soup.find("table")
    
    if table:
        # Go through each row in the table (skip the header row)
        for row in table.find_all("tr")[1:]:
            columns = row.find_all(["td", "th"])
            if len(columns) >= 2:
                # First column = variable code, Second column = description
                code = columns[0].get_text(strip=True)
                description = columns[1].get_text(strip=True)
                var_labels[code] = description
    
    print(f"Scraped {len(var_labels):,} variable labels from the Census website")

except Exception as e:
    print(f"Could not scrape labels (not critical): {e}")


# Show the first 5 rows of raw data
df_raw.head()

Downloaded 3,222 rows and 28 columns
Scraped 28,193 variable labels from the Census website


,NAME,B16004_025E,B16004_026E,B16004_027E,B16004_028E,B16004_029E,B16004_030E,B16004_031E,B16004_032E,B16004_033E,B16004_034E,B16004_035E,B16004_036E,B16004_037E,B16004_038E,B16004_039E,B16004_040E,B16004_041E,B16004_042E,B16004_043E,B16004_044E,B16004_045E,B14001_002E,B14001_008E,B19013_001E,B05012_003E,state,county
0,"Autauga County, Alabama",34424,646,514,97,35,0,273,240,33,0,0,315,182,99,16,18,161,7,37,117,0,13743,2430,68315,1552,01,001
1,"Baldwin County, Alabama",127387,4350,2039,1219,785,307,2096,1459,441,174,22,605,283,161,161,0,87,80,7,0,0,47838,7417,71039,8106,01,003
2,"Barbour County, Alabama",13587,958,597,106,226,29,97,69,0,28,0,97,47,20,30,0,61,30,31,0,0,5258,841,39712,776,01,005
3,"Bibb County, Alabama",13610,161,93,7,61,0,7,7,0,0,0,24,0,24,0,0,0,0,0,0,0,4488,461,50669,244,01,007
4,"Blount County, Alabama",31504,3053,1359,649,706,339,144,144,0,0,0,8,8,0,0,0,13,4,0,9,0,12727,1754,57440,2843,01,009


---
## Step 4: Clean and organise the data

The raw data has cryptic column names like `B16004_025E`. We need to:
1. **Rename columns** to something readable (e.g., `spanish_speakers_total`)
2. **Split** the "County, State" column into two separate columns
3. **Remove** unnecessary columns (like FIPS codes the API adds automatically)
4. **Convert** text numbers to actual numbers so we can do math on them
5. **Handle** missing values (the Census uses `-666666666` to mean "no data")
6. **Encode** state names as numbers (useful if we want to do machine learning later)

In [4]:
# ── STEP 4: Clean and organise the data ──


# --- 4a: Rename the cryptic column codes to readable names ---

# This dictionary maps each Census code to a clear English name
RENAME_MAP = {
    "NAME":          "county_state",  # Will be split into county + state later
    
    "B16004_025E":   "total_non_english_speakers_18_64",
    
    # Spanish speakers
    "B16004_026E":   "spanish_speakers_total",
    "B16004_027E":   "spanish_eng_very_well",
    "B16004_028E":   "spanish_eng_well",
    "B16004_029E":   "spanish_eng_not_well",
    "B16004_030E":   "spanish_eng_not_at_all",
    
    # Indo-European language speakers (French, Hindi, Russian, Portuguese, etc.)
    "B16004_031E":   "indo_euro_speakers_total",
    "B16004_032E":   "indo_euro_eng_very_well",
    "B16004_033E":   "indo_euro_eng_well",
    "B16004_034E":   "indo_euro_eng_not_well",
    "B16004_035E":   "indo_euro_eng_not_at_all",
    
    # Asian / Pacific Island speakers (Mandarin, Korean, Vietnamese, Tagalog, etc.)
    "B16004_036E":   "asian_pac_speakers_total",
    "B16004_037E":   "asian_pac_eng_very_well",
    "B16004_038E":   "asian_pac_eng_well",
    "B16004_039E":   "asian_pac_eng_not_well",
    "B16004_040E":   "asian_pac_eng_not_at_all",
    
    # Other language speakers (Arabic, Somali, Swahili, etc.)
    "B16004_041E":   "other_lang_speakers_total",
    "B16004_042E":   "other_lang_eng_very_well",
    "B16004_043E":   "other_lang_eng_well",
    "B16004_044E":   "other_lang_eng_not_well",
    "B16004_045E":   "other_lang_eng_not_at_all",
    
    # School enrollment
    "B14001_002E":   "total_enrolled_in_school",
    "B14001_008E":   "enrolled_college_or_grad_school",
    
    # Socioeconomic
    "B19013_001E":   "median_household_income_usd",
    "B05012_003E":   "foreign_born_population",
}

# Apply the renaming
df = df_raw.rename(columns=RENAME_MAP)

print("Columns renamed.")


# --- 4b: Split "County, State" into two separate columns ---

# The original NAME column looks like: "Los Angeles County, California"
# We want separate "county" and "state" columns

# First, extract county name (everything before the comma)
county_names = df_raw["NAME"].str.split(", ").str[0]

# Then, extract state name (everything after the comma)
state_names = df_raw["NAME"].str.split(", ").str[1]

# Remove the old combined column
df.drop(columns=["county_state"], inplace=True)

# Insert county as the first column and state as the second column
df.insert(0, "county", county_names)
df.insert(1, "state", state_names)

print("County and State columns created.")


# --- 4c: Remove unnecessary columns ---

# The Census API automatically adds FIPS code columns ("state" and "county" as numbers)
# We need to identify and remove these extra columns
# Keep only columns that are either "county", "state", or in our RENAME_MAP
columns_we_want = ["county", "state"] + list(RENAME_MAP.values())
columns_we_want.remove("county_state")  # We already removed this one

# Keep only the columns we want
columns_to_keep = [col for col in df.columns if col in columns_we_want]
df = df[columns_to_keep]

print("Unnecessary FIPS code columns removed.")


# --- 4d: Convert text to numbers ---

# The Census API returns everything as text (strings), even numbers
# We need to convert the numeric columns so we can do calculations
numeric_columns = [col for col in df.columns if col not in ["county", "state"]]

for col in numeric_columns:
    # pd.to_numeric converts text to numbers; errors="coerce" turns bad values into NaN
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("All numeric columns converted from text to numbers.")


# --- 4e: Handle the Census "no data" marker ---

# The Census uses the number -666666666 to mean "data not available"
# We replace it with NaN (pandas' way of saying "missing")
df.replace(-666666666, pd.NA, inplace=True)

print("Missing data markers (-666666666) replaced with NaN.")


# --- 4f: Convert state names to numbers (for potential machine learning use) ---

# LabelEncoder assigns a unique number to each state
# For example: "Alabama" = 0, "Alaska" = 1, "Arizona" = 2, etc.
encoder = LabelEncoder()
state_as_numbers = encoder.fit_transform(df["state"].fillna("Unknown"))

# Add this as a new column right after "state"
df.insert(2, "state_encoded", state_as_numbers)

print("State names encoded as numbers.")
print()
print(f"Final cleaned data: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Show the first 5 rows to verify everything looks correct
df.head()

Columns renamed.


ValueError: cannot insert county, already exists

---
## Step 5: Quick summary of the dataset

Before making charts, let's check the basic stats of our data.

In [ ]:
# ── STEP 5: Quick summary ──

print("DATASET SUMMARY")
print(f"  Total rows (counties):  {len(df):,}")
print(f"  Total columns:          {len(df.columns)}")
print(f"  Unique states:          {df['state'].nunique()}")
print(f"  Total missing values:   {df.isnull().sum().sum():,}")

---
## Step 6: Charts and Visualisations (EDA)

EDA stands for **Exploratory Data Analysis** — we create charts to understand the data visually.

### Chart 1: English Proficiency Distribution
How well do non-English speakers in the U.S. speak English?

In [ ]:
# ── CHART 1: English Proficiency Bar Chart ──

# For each proficiency level, we add up ALL language groups across ALL counties
# This gives us the total number of people at each English level

# "Very Well" = Spanish very well + Indo-European very well + Asian/Pac very well + Other very well
very_well_total = (
    df["spanish_eng_very_well"].sum() +
    df["indo_euro_eng_very_well"].sum() +
    df["asian_pac_eng_very_well"].sum() +
    df["other_lang_eng_very_well"].sum()
)

# Same calculation for "Well"
well_total = (
    df["spanish_eng_well"].sum() +
    df["indo_euro_eng_well"].sum() +
    df["asian_pac_eng_well"].sum() +
    df["other_lang_eng_well"].sum()
)

# Same for "Not Well"
not_well_total = (
    df["spanish_eng_not_well"].sum() +
    df["indo_euro_eng_not_well"].sum() +
    df["asian_pac_eng_not_well"].sum() +
    df["other_lang_eng_not_well"].sum()
)

# Same for "Not at All"
not_at_all_total = (
    df["spanish_eng_not_at_all"].sum() +
    df["indo_euro_eng_not_at_all"].sum() +
    df["asian_pac_eng_not_at_all"].sum() +
    df["other_lang_eng_not_at_all"].sum()
)

# Put the results into a small table for charting
proficiency_data = pd.DataFrame({
    "Proficiency Level": ["Very Well", "Well", "Not Well", "Not at All"],
    "Total People":      [very_well_total, well_total, not_well_total, not_at_all_total]
})

# Create the bar chart
fig = px.bar(
    proficiency_data,
    x="Proficiency Level",
    y="Total People",
    color="Proficiency Level",
    color_discrete_sequence=["#2ecc71", "#3498db", "#e67e22", "#e74c3c"],  # Green, Blue, Orange, Red
    title="English Proficiency Distribution — All Non-English Speakers (18–64)"
)

# Add text labels on top of each bar showing the count in millions
fig.update_traces(
    text=[f"{count / 1_000_000:.1f}M" for count in proficiency_data["Total People"]],
    textposition="outside",
    showlegend=False
)

# Style the chart
fig.update_layout(
    xaxis_title="English Proficiency Level",
    yaxis_title="Total Population",
    template="simple_white"
)

fig.show()

### Chart 2: Language Groups Pie Chart
What languages do non-English speakers in the U.S. speak?

In [ ]:
# ── CHART 2: Language Groups Pie Chart ──

# Add up the total speakers for each language group across all counties
spanish_total   = df["spanish_speakers_total"].sum()
indo_euro_total = df["indo_euro_speakers_total"].sum()
asian_pac_total = df["asian_pac_speakers_total"].sum()
other_total     = df["other_lang_speakers_total"].sum()

# Create the pie chart
fig = px.pie(
    values=[spanish_total, indo_euro_total, asian_pac_total, other_total],
    names=["Spanish", "Indo-European", "Asian/Pacific Island", "Other Languages"],
    color_discrete_sequence=["#e74c3c", "#3498db", "#f39c12", "#9b59b6"],  # Red, Blue, Yellow, Purple
    title="Non-English Language Groups (18–64)"
)

# Show the label and percentage inside each pie slice
fig.update_traces(
    textposition="inside",
    textinfo="label+percent",
    textfont=dict(size=11, color="white")
)

fig.show()

### Chart 3: Top 15 States by Foreign-Born Population
Which U.S. states have the most people born outside the country?

In [ ]:
# ── CHART 3: Top 15 States by Foreign-Born Population ──

# Group the data by state and add up the foreign-born population for each state
state_foreign_born = df.groupby("state")["foreign_born_population"].sum()

# Sort from highest to lowest and take the top 15
top_15_states = state_foreign_born.sort_values(ascending=False).head(15)

# Convert to a DataFrame for plotting
top_15_df = top_15_states.reset_index()

# Create a horizontal bar chart
fig = px.bar(
    top_15_df,
    y="state",                        # State names on the Y-axis
    x="foreign_born_population",      # Population count on the X-axis
    orientation="h",                   # Horizontal bars
    color="foreign_born_population",   # Color intensity based on population
    color_continuous_scale="Reds",     # Red color gradient
    title="Top 15 States by Foreign-Born Population",
    labels={
        "foreign_born_population": "Foreign-Born Population",
        "state": ""
    }
)

# Format the X-axis numbers with commas (e.g., 10,000,000 instead of 10000000)
fig.update_xaxes(tickformat=",.0f")

fig.show()

---
## Step 7: Export the cleaned data to a CSV file

CSV (Comma-Separated Values) is a simple file format that can be opened in Excel, Google Sheets, or any data tool. We save our cleaned data so we can reuse it without running the API again.

In [ ]:
# ── STEP 7: Export to CSV ──

# Define the order of columns in our final CSV file
column_order = [
    # Identity columns
    "county", "state", "state_encoded",
    
    # Population
    "total_non_english_speakers_18_64",
    "foreign_born_population",
    
    # Spanish speakers
    "spanish_speakers_total",
    "spanish_eng_very_well", "spanish_eng_well",
    "spanish_eng_not_well", "spanish_eng_not_at_all",
    
    # Indo-European speakers
    "indo_euro_speakers_total",
    "indo_euro_eng_very_well", "indo_euro_eng_well",
    "indo_euro_eng_not_well", "indo_euro_eng_not_at_all",
    
    # Asian / Pacific Island speakers
    "asian_pac_speakers_total",
    "asian_pac_eng_very_well", "asian_pac_eng_well",
    "asian_pac_eng_not_well", "asian_pac_eng_not_at_all",
    
    # Other language speakers
    "other_lang_speakers_total",
    "other_lang_eng_very_well", "other_lang_eng_well",
    "other_lang_eng_not_well", "other_lang_eng_not_at_all",
    
    # Social integration indicators
    "total_enrolled_in_school",
    "enrolled_college_or_grad_school",
    "median_household_income_usd",
]

# Only keep columns that actually exist in our data
final_columns = [col for col in column_order if col in df.columns]
df_final = df[final_columns].copy()

# Remove rows where ALL English proficiency columns are empty
# (These are counties with no proficiency data at all — not useful)
proficiency_columns = [col for col in df_final.columns if "_eng_" in col]
df_final.dropna(subset=proficiency_columns, how="all", inplace=True)

# Reset the row numbering (0, 1, 2, 3, ...)
df_final.reset_index(drop=True, inplace=True)

# Save to CSV file
OUTPUT_FILENAME = "lang_proficiency_social_integration_counties.csv"
df_final.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")

print(f"File saved: {OUTPUT_FILENAME}")
print(f"Total rows: {len(df_final):,}")
print(f"Total columns: {len(df_final.columns)}")

# Show the first 5 rows of the final file
df_final.head()